## Monitorage de l'utilisation des ressources

Il est essentiel de veiller à ce que vos tâches utilisent efficacement les ressources qui leur sont allouées. Cela est particulièrement vrai lorsque vous utilisez un nouveau programme ou que vous avez apporté une modification importante au travail effectué par votre tâche. Ce chapitre décrit différentes méthodes d'évaluation de l'efficacité des tâches, qu'elles soient en cours d'exécution ou terminées.

### Monitorage des tâches via la ligne de commande

#### Taches actuels
Par défaut, la file d'attente affichera toutes les tâches gérées par le planificateur. Elle sera beaucoup plus rapide si vous ne consultez que vos propres tâches.

`squeue -u $USER`

Vous pouvez afficher uniquement les tâches en cours ou uniquement les tâches en attente :

`squeue -u <username> -t RUNNING` <br>
`squeue -u <username> -t PENDING`

Vous pouvez afficher des informations détaillées pour un emploi spécifique avec scontrol :

`scontrol show job -dd JOB_ID`

N’exécutez pas la commande squeue à haute fréquence (par exemple, toutes les quelques secondes) depuis un script ou un programme. Répondre à squeue surcharge Slurm et peut nuire à ses performances ou à son bon fonctionnement.

#### Taches terminés
Vous pouvez obtenir un bref résumé de l'efficacité du processeur et de la mémoire d'une tâche avec `seff` :

`seff JOB_ID`

Des informations plus détaillées sur un travail terminé peuvent être obtenues avec `sacct` :

`sacct -j JOB_ID`

#### Se connecter à une tâche en cours

Il est possible de se connecter au nœud exécutant une tâche et d'y lancer de nouveaux processus. Cela peut s'avérer utile pour le dépannage ou le suivi de la progression d'une tâche.

Supposons que vous souhaitiez exécuter l'utilitaire nvidia-smi pour surveiller l'utilisation du GPU sur un nœud où une tâche est en cours d'exécution. La commande suivante lance une surveillance sur le nœud affecté à la tâche, ce qui exécute nvidia-smi toutes les 30 secondes et affiche le résultat dans votre terminal :

`srun --jobid 31 --overlap --pty watch -n 30 nvidia-smi`

Il est possible de lancer plusieurs commandes de surveillance avec tmux. La commande suivante lance htop et nvidia-smi dans des fenêtres séparées pour surveiller l'activité sur un nœud affecté à la tâche spécifiée :

`srun --jobid 31 --overlap --pty tmux new-session -d 'htop -u $USER' \; split-window -h 'watch nvidia-smi' \; attach`

**Attention :** les processus lancés avec `srun` partagent les ressources avec la tâche spécifiée. Il convient donc de veiller à ne pas lancer de processus susceptibles d'utiliser une part importante des ressources allouées à la tâche. Par exemple, une utilisation excessive de la mémoire peut entraîner l'arrêt brutal de la tâche ; une utilisation excessive du processeur la ralentira. Supposons que vous souhaitiez exécuter l'utilitaire `nvidia-smi` pour surveiller l'utilisation du GPU sur un nœud où une tâche est en cours d'exécution. La commande suivante lance `watch` sur le nœud affecté à la tâche, ce qui exécute à son tour `nvidia-smi` toutes les 30 secondes et affiche le résultat dans votre terminal :

### Monitorage des processus CPU
#### Top
La commande `top` est un outil de surveillance système en temps réel préinstallé sur la plupart des distributions Linux. Elle offre une vue dynamique des ressources système, notamment une liste en direct des processus en cours d'exécution.

**Start Monitoring**: Ouvrez votre terminal et tapez top, puis appuyez sur Entrée. <br>
**Exit**: Appuyez sur q pour quitter l'interface et revenir à la ligne de commande. 

![cpu-vs-cpu-bandwitdh](images/top.png)

<u>*Les métriques du CPU*</u>

%Cpu(s) Line: <br>
- us (User): Time spent on user-level processes (e.g., applications).
- sy (System): Time spent on kernel processes.
- id (Idle): Time the CPU is doing nothing. High idle means low load.
- wa (I/O Wait): Time the CPU spends waiting for disk or network operations.

<u>*Options de démarrage spécialisées*</u>

You can launch top with specific parameters for automation or targeted monitoring: 
- Monitor a specific user: top -u [username].
- Monitor specific PIDs: top -p [PID1],[PID2].
- Batch Mode (Snapshots): Use top -b -n 1 > snapshot.txt to capture a single "snapshot" of system state to a file without the interactive interface.

#### Htop

Htop est un système de monitorage interactif en temps réel et un gestionnaire de processus pour les systèmes de type Unix. Il constitue une alternative améliorée et conviviale à la commande top traditionnelle. Il affiche l'utilisation du processeur et de la mémoire, fournit des indicateurs de performance codés par couleur et permet de gérer les processus (kill, renice) à l'aide des touches fléchées et des touches de fonction.


![cpu-vs-cpu-bandwitdh](images/htop.png)

<u>*Key Interactive Features*</u>

- Navigation: Use arrow keys to scroll vertically/horizontally through processes.
- Sorting: Press F6 to sort by CPU, memory, or PID.
- Search: Press F3 to search for a specific process.
- Manage Processes: Press F9 to kill or send signals to a highlighted process.
- Tree View: Press F5 to show processes in a hierarchical tree format.
- Colors: Color-coded bars provide quick visual status updates.

### Monitorage des processus GPU
#### Nvidia-smi

Nous allons commencer par les outils de surveillance. La première chose à vérifier est si votre code utilise bien le GPU. La méthode la plus simple pour répondre à cette question est d'utiliser la commande `nvidia-smi`.

![cpu-vs-cpu-bandwitdh](images/nvidia-smi.png)

Comme vous pouvez le voir ci-dessus, <code>nvidia-smi</code> vous indiquera si des processus utilisent actuellement le GPU et quelle quantité de mémoire GPU ils ont allouée.

Une erreur fréquente d'interprétation du champ <code>utilisation %</code> consiste à le considérer comme la quantité de capacité de calcul actuellement utilisée. Or, ce n'est **pas** ce que représente ce nombre. En réalité, <code>utilisation %</code> correspond à la proportion de temps pendant laquelle *au moins un noyau* était exécuté sur le GPU. Un faible pourcentage signifie que le GPU est soit inactif, soit occupé à d'autres tâches que l'exécution de noyaux, comme la gestion de la mémoire ou la planification des tâches.


Idéalement, ce pourcentage reste élevé pendant toute la durée de votre tâche, mais **cela ne suffit pas** à conclure que votre code n'utilise pas inutilement la capacité du GPU. Vous pourriez en effet avoir un seul noyau fonctionnant en permanence avec seulement quelques cœurs, gaspillant ainsi massivement la capacité du GPU.

#### Nvtop

Another way of checking whether or not your code is actually using the GPU is the command <code>nvtop</code>. This tool will display the same usage statistics as <code>nvidia-smi</code> as a line plot that changes over time. You will also get a breakdown of <code>utilisation %</code> per process if you have multiple processes sharing the same GPU:

![cpu-vs-cpu-bandwitdh](images/nvtop.png)

The line plot is useful to identify peaks and cliffs in <code>utilisation %</code> as well as any changes in memory allocation during the execution of your code. This can provide helpful clues for you to optimize your code to minimize idle time for example.

Another interesting number provided by this tool is the **CPU % utilisation**. In general, CPU usage should be low in a GPU program. A high CPU utilisation % might indicate your program is not using the GPU optimally as a lot of work is being done by the CPU.

### Monitorage des ressources avec des portails des grappes de calul
Nous allons maintenant nous intéresser à un outil qui se situe à la frontière entre la surveillance et le profilage : les portails de regroupement de l’Alliance de recherche numérique du Canada.

<u>*Monitoring CPU metrics*</u>
![cpu-vs-cpu-bandwitdh](images/portal1.png)

![cpu-vs-cpu-bandwitdh](images/portal2.png)

<u>*Monitoring whole node resources*</u>
![cpu-vs-cpu-bandwitdh](images/portal3.png)

<u>*Monitoring GPU metrics*</u>

![cpu-vs-cpu-bandwitdh](images/portal.png)


Above you can see a different set of statistics pertaining to the same code example:
- **CPU:** This is the average of the ratio of cycles consumed on each CPU core by all processes in this job. This graph should be all filled up most of the time, if not, you can lower the cores requested to the scheduler. Unused cycles does not improve your job performance and will lower your priority when cores are wasted.

- **Memory:** This is the max used memory should be close to the allocated memory. If the memory is not used by the job, ask a lower amount, your jobs will be able to start faster. Unused memory does not increase your job performance.

- **Filesystem:** This is the filesystem usage of the job, not the whole node.

- **SM Activity:** This is the average % of time over all SMs where there was at least one set of concurrent operations active, no matter how many kernels that happens to be. Note that *active* does not mean *computing*. Other activities such as waiting for data and acessing memory also count here. Ideally this number should stay over 80% for the duration of your job, but that **does not necessarily mean** you are using 80% of the GPUs compute capacity. It means that either 80% of the time your were using 100% of the capcity and stayed idle 20% of the time, or that you used 80% capacity 100% of the time with 20% of vacant capacity.

<br>

- **SM Occupancy:**  This is the average, over time, of the ratio of active concurrent operations and the maximum number of concurrent operations supported by the GPU. A high % also **does not necessarily mean** you are using the GPU effectively. For our purposes however, we will say that this number being high is a strong indicator of good usage.

<br>

The other metrics we see above - <code>Tensor</code>, <code>FP64</code>, <code>FP32</code> and <code>FP16</code> - indicate what type of GPU core is being used and what % of time these specific parts of the card were active. Of note here is the <code>Tensor</code> metric. Tensor cores are specialized cores purpose built for speeding-up n-dimensional tensor operations like multiplications and convolutions. If your code makes heavy use of tensor operations, the Cuda compiler can automatically take advantage of these specialized cores if your program satisfies one of these conditions:

- You use [mixed precision](https://en.wikipedia.org/wiki/Mixed-precision_arithmetic) in your program.
- You explicitly use [Nvidia's TF32 data type](https://blogs.nvidia.com/blog/2020/05/14/tensorfloat-32-precision-format/) in your program.

A high % usage of Tensor cores should correspond to good speed-ups relative to other GPU core types. One caveat is that the first two conditions require sacrificing precision in computations, so you have to be mindful of whether or not high precision is important for your application.

The alliance currently maintains cluster portals for the following clusters:

- [Rorqual](https://metrix.rorqual.calculquebec.ca)
- [Narval](https://portail.narval.calculquebec.ca)
- [Nibi](https://portal.nibi.sharcnet.ca)
- [Tamia](https://portail.tamia.ecpia.ca)
- [Trillium](https://my.scinet.utoronto.ca)

# Example:  CIFAR10 data set
We use this example to show you how to monitor CPU/GPU resources using portals as well as simple tools such as top,htop,nvtop, and nvidia-smi. 

You will use PyTorch to fit a Convolutional Neural Network to the CIFAR10 dataset - a collection of 60000 images representing 10 different animals and transportation methods.
The convolutional neural network consists of a sequence of two convolutions, followed by 400 concurrent matrix multiplications, then 120 concurrent matrix multiplications and finally 10 concurrent matrix multiplications. Both convolutions and matrix multiplications are operations that can be parallelized. 


In [1]:
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

device = torch.device("cuda")

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

#net = Net().cuda() # Load model on the GPU
net = Net().to(device)

#criterion = nn.CrossEntropyLoss().cuda() # Load the loss function on the GPU
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(net.parameters(), lr=0.001)

transform_train = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

Here we use a variant of the gradient descent method called mini-batch gradient descent. Instead of using the entire matrix of inputs at each iteration of the algorithm, you use a small subset of the rows. The amount of rows that make up this subset, called the batch size, is an important factor that affects whether or not as well as how quickly this approach converges.

Next, you will use mini-batch gradient descent to fit our convolutional neural network to the CIFAR10 data. You will use the code below to load one batch of images at a time from disk, then time the execution of 3 iterations of this training algorithm. Try it out with different combinations of the three parameters below and see what the impact is on execution time and on GPU usage:

1. **BATCH_SIZE**: this is the mini-batch gradient descent batch size, i.e., how many images should be loaded from disk and used by the trainig algorith at a time.
2. **NUM_WORKERS**: this controls how many CPUs will be used to load batches of images *in parallel*. The idea is to use multiple processes, with one CPU each, to streamline loading images from disk *without* blocking the main process.
3. **PRE_FETCH**: this controls how many batches of images should be loaded from disk *before* they are needed by the training algorithm. These batches will be stored in the system's memory and loaded on the GPU once its their turn to be used.

In [2]:
f"{os.getenv('SLURM_TMPDIR')}/data"

'/localscratch/instructor3.106.0/data'

In [3]:
BATCH_SIZE = 4096
NUM_WORKERS = 2
PRE_FETCH = 2

#datadir = os.environ["CIFAR10_PATH"] #f"{os.getenv('SLURM_TMPDIR')}/data"
datadir = f"{os.getenv('SLURM_TMPDIR')}/data"
dataset_train = CIFAR10(root=datadir, train=True, download=False, transform=transform_train)

train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, prefetch_factor=PRE_FETCH)

In [4]:
def train(model, n_iterations, data):
    for iteration in range(n_iterations):
        for batch_idx, (inputs, targets) in data:

#            inputs = inputs.cuda()
#            targets = targets.cuda()
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [5]:
%time train(net,n_iterations=3, data=enumerate(train_loader))

CPU times: user 535 ms, sys: 358 ms, total: 893 ms
Wall time: 5.71 s


In the example above, we used the <code>DataLoader</code> object to iretatively read bacthes of images from disk, which means 2 sets of read/write operations: <code>disk -> system memory</code> and <code>system memory -> GPU memory</code>. The <code>disk -> system memory</code> step is prone to causing bottlenecks, as reading from disk is usually much slower than reading from memory. What would the difference in performance be if we instead loaded the whole dataset to the system memory before starting our computations? 

In [6]:
cifar10 = [(batch_idx,(inputs,targets)) for batch_idx, (inputs, targets) in enumerate(train_loader)]

%timeit train(net,n_iterations=3, data=cifar10)

2.37 s ± 108 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


The convolution neural network you used above was, relatively speaking, not that big. If you look at how much GPU memory is used when that model is loaded, you will see it is somewhere between 1 and 2 GB. There is more than enough space left to load the CIFAR10 dataset. Could we have sped-up the training algorithm even more simply by loading all images in GPU memory before calling <code>train()</code>?

In [7]:
cifar10 = [(batch_idx,(inputs.cuda(),targets.cuda())) for batch_idx, (inputs, targets) in enumerate(train_loader)]

%timeit train(net,n_iterations=3, data=cifar10)

1.84 s ± 327 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Matrix multiplication
Lets run the matrix multiplication example as it's very GPU intensive task and monitor GPU usage on the portal.

In [2]:
import os
import timeit

N_CPUS=4

os.environ["OMP_NUM_THREADS"] = str(N_CPUS)

import numpy as np
import cupy as cp

In [1]:
import cupy
print(cupy.cuda.runtime.runtimeGetVersion())

12090


In [3]:
pip install --no-index cupy==13.6.0

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic/cupy-13.6.0+computecanada-cp312-cp312-linux_x86_64.whl
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/fastrlock-0.8.3+computecanada-cp312-cp312-linux_x86_64.whl (from cupy==13.6.0)
  Attempting uninstall: cupy
    Found existing installation: cupy 14.1.0+computecanada
    Uninstalling cupy-14.1.0+computecanada:
      Successfully uninstalled cupy-14.1.0+computecanada
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cupy]1/2 [cupy]
Note: you may need to restart the kernel to use updated packages.


In [3]:
def generate_data(n_rows,n_cols,device):
    lib = cp if device == "gpu" else np

    rmg = lib.random.rand #random matrix generator
    a = rmg(n_rows,n_cols)
    b = rmg(n_rows,n_cols)
    return a,b

def matmul(a,b):
    lib = cp if type(a) == cp.ndarray else np

    results = lib.matmul(a,b)
    return

In [4]:
a,b = generate_data(5000,5000,"gpu")

In [8]:
%timeit matmul(a,b)

102 ms ± 10.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
